# Membership Inference Attacks for DP-GAN & DP-CTGAN

Author: Ilse Harmers \
Last modified: March 19, 2026

Membership inference attack pipeline is based on the descriptions provided by:

    Stadler, T., Oprisanu, B., & Troncoso, C. (2020). Synthetic data -- Anonymisation Groundhog Day. arXiv (Cornell University). https://doi.org/10.48550/arxiv.2011.07018

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from snsynth import Synthesizer
import snsynth.transform as tf
import utils

import torch
device = torch.device("cuda:0" if (torch.cuda.is_available() and 1 > 0) else "cpu")
print(device)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

In [ ]:
# Setting up the dataset information and framework's hyperparameters.
data_info = {"adult" : {"folder": "adult"}, "bank": {"folder": "bank-marketing"}, "credit" : {"folder": "credit-card-default"}}
dataset_name = "adult"
folder = data_info[dataset_name]["folder"]
train_full = pd.read_csv(f"../train-test-datasets/{folder}/{dataset_name}_train.csv")   # importing train set

cat_cols = train_full.select_dtypes(include=['object']).columns.to_list()
num_cols = train_full.select_dtypes(exclude=['object']).columns.to_list()
print(cat_cols, num_cols)

# Defining epsilon value.
epsi = 5

# Defining differentially private GAN model: [dpgan, dpctgan].
model = "dpctgan"
if model == "dpgan":
    folder1 = "DP-GAN"
elif model == "dpctgan":
    folder1 = "DPCTGAN"
else:
    raise ValueError("Model not supported.")

ref_size = 20000
n_size = 2000
m_size = 400
batches = 750
runs = 15

# Dictionary containing list of target outliers' indices for each of our datasets.
if dataset_name == "adult":
    targets = {"adult": [train_full.loc[train_full["capital-loss"] == 4356.0].index, train_full.loc[train_full["hours-per-week"] == 99].head(1).index,
                         train_full.loc[train_full["capital-gain"] == 99999].head(1).index, train_full.loc[train_full["age"] == 89].index,
                         train_full.loc[train_full["education-num"] == 16].head(1).index]}
elif dataset_name == "bank":
    targets = {"bank": [train_full.loc[train_full["balance"] == 102127].index, train_full.loc[train_full["pdays"] == 871].index,
                        train_full.loc[train_full["duration"] == 4918].index, train_full.loc[train_full["campaign"] == 63].index,
                        train_full.loc[train_full["previous"] == 275].index,]}
elif dataset_name == "credit":
    targets = {"credit": [train_full.loc[train_full["LIMIT_BAL"] == 1000000].index, train_full.loc[train_full["PAY_AMT4"] == 621000].index,
                          train_full.loc[train_full["BILL_AMT2"] == 983931].index, train_full.loc[train_full["PAY_AMT5"] == 426529].index,
                          train_full.loc[train_full["AGE"] == 79].index]}
else: 
    raise ValueError("Dataset not supported.")

s_vals = [0, 1]

In [ ]:
#train_full.loc[train_full["hours-per-week"] == 99].index
#len(train_full.loc[train_full["hours-per-week"] == 99].index)
#97/36177   # 0.00268
#train_full.loc[train_full["capital-gain"] == 99999].index
#len(train_full.loc[train_full["capital-gain"] == 99999].index)
#185/36177   # 0.00511
#train_full.loc[train_full["education-num"] == 16].index
#len(train_full.loc[train_full["education-num"] == 16].index)
#444/36177   # 0.01227
#print(train_full["age"].quantile(0.95))   # 62
#print(train_full["capital-loss"].quantile(0.95))   # 0.0

#print(train_full["balance"].quantile(0.95))   # 5781
#print(train_full["pdays"].quantile(0.95))   # 317
#print(train_full["duration"].quantile(0.95))   # 744
#print(train_full["campaign"].quantile(0.95))   # 8
#print(train_full["previous"].quantile(0.95))   # 3

#print(train_full["LIMIT_BAL"].quantile(0.95))   # 430000
#print(train_full["PAY_AMT4"].quantile(0.95))   # 16490 
#print(train_full["BILL_AMT2"].quantile(0.95))   # 194304
#print(train_full["PAY_AMT5"].quantile(0.95))   # 16000
#print(train_full["AGE"].quantile(0.95))   # 53

#train_full.loc[train_full["AGE"] == 79]
train_full.loc[train_full["age"] == 95]

In [ ]:
train_full.describe()

In [ ]:
train_full.describe()

In [ ]:
# Setting up preprocessor table transformers for the Adult, Bank and Credit datasets.

def adult_tf():
    """This function provides an Adult-based table transformer."""

    tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = 17, upper = 90, 
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # workclass
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education-num
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital-status
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # occupation
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # relationship
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # race
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # sex
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = np.log(1), upper = np.log(99999.0 + 1),
                             negative = False) # capital-gain; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = np.log(1), upper = np.log(4400.0 + 1),
                             negative = False) # capital-loss; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = 1, upper = 99, negative = False), # hours-per-week; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # native-country
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # income
    ])

    return tt

def bank_tf():
    """This function provides a Bank-based table transformer."""

    tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = 18, upper = 95,
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # job
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # default
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-8000) + 1)), upper = np.log(102000 + 1), 
                             negative = False) # balance; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # housing
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # loan
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # contact
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # day
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # month
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = np.log(1), upper = np.log(4900 + 1), 
                             negative = False) # duration; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = 1, upper = 60,
                         negative = False), # campaign; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-1) + 1)), upper = np.log(870 + 1),
                             negative = False) # pdays; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = np.log(1), upper = np.log(280 + 1),
                             negative = False) # previous; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # poutcome
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # y
    ])

    return tt

def credit_tf():
    """This function provides a Credit-based table transformer."""

    tt = tf.TableTransformer([
    tf.ChainTransformer([
        tf.LogTransformer(),
        tf.MinMaxTransformer(lower = np.log(10000.0), upper = np.log(1000000.0),
                             negative = False) # LIMIT_BAL; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # SEX
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # EDUCATION
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # MARRIAGE
    tf.MinMaxTransformer(lower = 21, upper = 79, negative = False), # AGE; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_0
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_2
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_3
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_4
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_5
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_6
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-166000.0) + 1)), 
                             upper = np.log(965000.0 + 1), negative = False) # BILL_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-70000.0) + 1)), 
                             upper = np.log(984000.0 + 1), negative = False) # BILL_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-157000.0) + 1)), 
                             upper = np.log(1664000.0 + 1), negative = False) # BILL_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-170000.0) + 1)), 
                             upper = np.log(892000.0 + 1), negative = False) # BILL_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-81000.0) + 1)), 
                             upper = np.log(927000.0 + 1), negative = False) # BILL_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-340000.0) + 1)), 
                             upper = np.log(962000.0 + 1), negative = False) # BILL_AMT6; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(874000.0 + 1), 
                             negative = False) # PAY_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(1684000.0 + 1), 
                             negative = False) # PAY_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(896000.0 + 1), 
                             negative = False) # PAY_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(621000.0 + 1), 
                             negative = False) # PAY_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(427000.0 + 1), 
                             negative = False) # PAY_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(529000.0 + 1), 
                             negative = False) # PAY_AMT6; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # DEFAULT
    ])

    return tt

In [ ]:
target_results = {}
for t in range(len(targets[dataset_name])):

    # We need to drop our target from the full training set before sampling.
    train_wo_t = train_full.drop(targets[dataset_name][t]).sample(n = ref_size, random_state = 42)
    
    for s in s_vals:
    
        r = 1
        while r < runs + 1:
    
            print(f"Run {r}")

            if s == 0:   # training without target
                train = train_wo_t.sample(n = n_size, random_state = r)
            elif s == 1:   # training with target
                train_sub = train_wo_t.sample(n = n_size - 1, random_state = r)
                train = pd.concat([train_sub, train_full.loc[targets[dataset_name][t]]])
            else:
                raise ValueError("'s' must be equal to 1 or 0.")
            #print(train)
            #print(train.shape)
            
            # Defining delta as the inverse of the size of the dataset. 
            delta = 10**np.floor(np.log10(1/train.shape[0]))
            print(delta)

            # Synthesizing the dataset with DP-GAN or DP-CTGAN. 
            synth = Synthesizer.create(model, epsilon = epsi, delta = delta, batch_size = 64, epochs = 10000,
                                       sigma = 3.5, verbose = False,   # (!) uncomment this line if model is DP-CTGAN
                                      )
            
            # Defining the table transformer.
            if dataset_name == "adult":
                tt = adult_tf()
            elif dataset_name == "bank":
                tt = bank_tf()
            elif dataset_name == "credit":
                tt = credit_tf()
            else:
                raise ValueError("Dataset not supported.")
                
            synth.fit(train, transformer = tt, preprocessor_eps = 0.0)

            # Saving the synthetic datasets.
            for i in range(int(batches/runs)):
                sample = synth.sample(m_size)
                sample.to_csv(f"./{folder1}/epsilon{epsi}/{folder}/target-record{t + 1}/s{s}/run{r}/sample{i + 1}.csv", index = False)
        
            r += 1

    # Initializing new column names for numerical and categorical columns. 
    names_mean = [f"{name}-mean" for name in num_cols]
    names_med = [f"{name}-med" for name in num_cols]
    names_var = [f"{name}-var" for name in num_cols]
    if cat_cols:
        names_hfreq = [f"{name}-hf" for name in cat_cols]
        names_lfreq = [f"{name}-lf" for name in cat_cols]
        names_len = [f"{name}-len" for name in cat_cols]

    if cat_cols:
        col_names = names_mean + names_med + names_var + names_hfreq + names_lfreq + names_len
    else:
        col_names = names_mean + names_med + names_var
    #print(len(col_names))
    #print(col_names)

    # Summarizing the synthetic datasets: every synthetic dataset is condensed to a single row of information on 
    # the amount and frequencies of categorical labels in categorical columns and the mean, median and variance of numerical columns. 
    dataset = {}
    a = 1
    for s in s_vals:
        for r in range(runs):
            for i in range(int(batches/runs)):
                sample = pd.read_csv(f"./{folder1}/epsilon{epsi}/{folder}/target-record{t + 1}/s{s}/run{r + 1}/sample{i + 1}.csv")
                if cat_cols:
                    hfreq = pd.Series({cat_cols[c]: sample.value_counts(cat_cols[c]).index[0] for c in range(len(cat_cols))})
                    lfreq = pd.Series({cat_cols[c]: sample.value_counts(cat_cols[c]).index[-1] for c in range(len(cat_cols))})
                    lens = pd.Series({cat_cols[c]: len(sample.value_counts(cat_cols[c]).index) for c in range(len(cat_cols))})
                    summary_res = pd.concat([sample[num_cols].mean(), sample[num_cols].median(), sample[num_cols].var(ddof = 0),
                                             hfreq, lfreq, lens])
                else:
                    summary_res = pd.concat([sample[num_cols].mean(), sample[num_cols].median(), sample[num_cols].var(ddof = 0)])
                dataset[f"sample{a}"] = {col_names[k]: summary_res.iloc[k] for k in range(len(col_names))}
                a += 1

    dataset = pd.DataFrame(dataset).T
    dataset["s"] = [0] * 750 + [1] * 750   # we have 30 runs * 50 synthetic datasets per run = 1500 rows of information on the synthetic datasets
    if cat_cols:
        dataset = dataset.astype({**{c: float for c in names_mean + names_med + names_var + names_len},
                                  **{s: str for s in names_hfreq + names_lfreq}})

    # Splitting the dataset into a train and test set with a 80:20 ratio.
    train_set = dataset.iloc[np.r_[0:600, 750:1350]]
    test_set = dataset.iloc[np.r_[600:750, 1350:1500]]

    # Preprocessing the train and test sets for an Ml classifier.
    if cat_cols:
        train_cat_encoded = utils.one_hot_encode(train_set[names_hfreq + names_lfreq])
        train_encoded = pd.concat([train_set[names_mean + names_med + names_var + names_len + ["s"]].reset_index(drop = True), 
                                   train_cat_encoded.reset_index(drop = True)], axis = 1)
        test_cat_encoded = utils.one_hot_encode(test_set[names_hfreq + names_lfreq])

        # Checking for missing columns between the train and test datasets.
        missing_cols = list(set(list(train_cat_encoded.columns)) - set(list(test_cat_encoded.columns)))
        # If there is a missing column, most likely due to a label from the training dataset not being present in the test set, then 
        # we reinsert a 'zero' column into the test set at the right place to account for this missing label.
        if missing_cols:
            for c in missing_cols:
                df_col = pd.DataFrame({c: test_set["age-mean"]*0})
                test_cat_encoded.insert(0, c, value = df_col)
        test_cat_encoded = test_cat_encoded[train_cat_encoded.columns]
        test_encoded = pd.concat([test_set[names_mean + names_med + names_var + names_len + ["s"]].reset_index(drop = True), 
                                  test_cat_encoded.reset_index(drop = True)], axis = 1)

        # Checking for missing columns between the test and train datasets.
        missing_cols = list(set(list(test_encoded.columns)) - set(list(train_encoded.columns)))
        # If there is a missing column, most likely due to a label from the test dataset not being present in the training set, then 
        # we reinsert a 'zero' column into the train set at the right place to account for this missing label.
        if missing_cols:
            for c in missing_cols:
                df_col = pd.DataFrame({c: test_set["age-mean"]*0})
                train_encoded.insert(0, c, value = df_col)
        train_encoded = train_encoded[test_encoded.columns]
        test_encoded = test_encoded[train_encoded.columns]
    else:
        train_encoded = train_set.copy()
        test_encoded = test_set.copy()
    #print(train_encoded.shape)

    np.random.seed(42)   # setting random seed for reproducibility. 
    # Fitting a random forest classifier and getting its predictions.
    rf = RandomForestClassifier()
    
    if dataset_name == "adult":
        if t == 0:
            train_rf = train_encoded[["capital-loss-mean", "capital-loss-med", "capital-loss-var"]]
            test_rf = test_encoded[["capital-loss-mean", "capital-loss-med", "capital-loss-var"]]
        elif t == 1:
            train_rf = train_encoded[["hours-per-week-mean", "hours-per-week-med", "hours-per-week-var"]]
            test_rf = test_encoded[["hours-per-week-mean", "hours-per-week-med", "hours-per-week-var"]]
        elif t == 2:
            train_rf = train_encoded[["capital-gain-mean", "capital-gain-med", "capital-gain-var"]]
            test_rf = test_encoded[["capital-gain-mean", "capital-gain-med", "capital-gain-var"]]
        elif t == 3:
            train_rf = train_encoded[["age-mean", "age-med", "age-var"]]
            test_rf = test_encoded[["age-mean", "age-med", "age-var"]]
        else:
            train_rf = train_encoded[["education-num-mean", "education-num-med", "education-num-var"]]
            test_rf = test_encoded[["education-num-mean", "education-num-med", "education-num-var"]]
    elif dataset_name == "bank":
        if t == 0:
            train_rf = train_encoded[["balance-mean", "balance-med", "balance-var"]]
            test_rf = test_encoded[["balance-mean", "balance-med", "balance-var"]]
        elif t == 1:
            train_rf = train_encoded[["pdays-mean", "pdays-med", "pdays-var"]]
            test_rf = test_encoded[["pdays-mean", "pdays-med", "pdays-var"]]
        elif t == 2:
            train_rf = train_encoded[["duration-mean", "duration-med", "duration-var"]]
            test_rf = test_encoded[["duration-mean", "duration-med", "duration-var"]]
        elif t == 3:
            train_rf = train_encoded[["campaign-mean", "campaign-med", "campaign-var"]]
            test_rf = test_encoded[["campaign-mean", "campaign-med", "campaign-var"]]
        else:
            train_rf = train_encoded[["previous-mean", "previous-med", "previous-var"]]
            test_rf = test_encoded[["previous-mean", "previous-med", "previous-var"]]
    elif dataset_name == "credit":
        if t == 0:
            train_rf = train_encoded[["LIMIT_BAL-mean", "LIMIT_BAL-med", "LIMIT_BAL-var"]]
            test_rf = test_encoded[["LIMIT_BAL-mean", "LIMIT_BAL-med", "LIMIT_BAL-var"]]
        elif t == 1:
            train_rf = train_encoded[["PAY_AMT4-mean", "PAY_AMT4-med", "PAY_AMT4-var"]]
            test_rf = test_encoded[["PAY_AMT4-mean", "PAY_AMT4-med", "PAY_AMT4-var"]]
        elif t == 2:
            train_rf = train_encoded[["BILL_AMT2-mean", "BILL_AMT2-med", "BILL_AMT2-var"]]
            test_rf = test_encoded[["BILL_AMT2-mean", "BILL_AMT2-med", "BILL_AMT2-var"]]
        elif t == 3:
            train_rf = train_encoded[["PAY_AMT5-mean", "PAY_AMT5-med", "PAY_AMT5-var"]]
            test_rf = test_encoded[["PAY_AMT5-mean", "PAY_AMT5-med", "PAY_AMT5-var"]]
        else:
            train_rf = train_encoded[["AGE-mean", "AGE-med", "AGE-var"]]
            test_rf = test_encoded[["AGE-mean", "AGE-med", "AGE-var"]]
    else:
        raise ValueError("Dataset not supported.")
        
    rf_model = rf.fit(train_rf, train_encoded["s"])
    preds_s = rf_model.predict(test_rf)

    # Determine confusion matrix of RF classifier.
    conf_mat = confusion_matrix(test_encoded["s"], preds_s)
    # Extract true positives (TP), true negatives (TN), false positives (FP) and false negatives (FN).
    tn, fp, fn, tp = conf_mat.ravel()
    tp_rate = tp/(tp + fn)   # true positive rate
    fp_rate = fp/(fp + tn)   # false positive rate
    PG = 1 - (tp_rate - fp_rate)

    target_results[f"target-record{t + 1}"] = PG

target_results = pd.DataFrame(target_results, index = ["PG"]).T
target_results.to_csv(f"./{folder1}/epsilon{epsi}/{folder}/target-results.csv")

In [ ]:
target_results